# EDA dan cleaning dataset Pokemon TCG

Notebook ini menganalisis seluruh metadata CSV dan audit gambar lokal dari manifest downloader. Tidak ada request HTTP dari notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'backend' / 'dataset').exists():
    root = root.parent.parent

dataset_dir = root / 'backend' / 'dataset'
csv_path = dataset_dir / 'pokemon_cards_dataset.csv'
json_path = dataset_dir / 'pokemon_cards_dataset.json'
image_dir = dataset_dir / 'pokemon-cards-dataset'
manifest_path = image_dir / 'image_manifest.csv'
clean_csv_path = dataset_dir / 'pokemon_cards_dataset_clean.csv'
clean_json_path = dataset_dir / 'pokemon_cards_dataset_clean.json'

df = pd.read_csv(csv_path)
print(f'Metadata: {len(df):,} baris, {len(df.columns)} kolom')
print(f'Folder gambar: {image_dir}')
print(f'Manifest tersedia: {manifest_path.exists()}')
display(df.head())

In [ ]:
missing_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'unique_count': df.nunique(dropna=True),
}).sort_values('missing_count', ascending=False)
display(missing_summary)
print(f"Duplicate card_id: {df['card_id'].duplicated().sum():,}")
print(f"Unique card_id: {df['card_id'].nunique():,}")

In [ ]:
def plot_top(column, title, limit=15):
    counts = df[column].fillna('[missing]').value_counts().head(limit)
    return counts.sort_values().plot(kind='barh', title=title, figsize=(8, 5))

display(plot_top('rarity', 'Top rarity'))
display(plot_top('types', 'Top tipe kartu'))
if 'supertype' in df.columns:
    display(df['supertype'].fillna('[missing]').value_counts().to_frame('count'))
else:
    print("Kolom 'supertype' tidak tersedia pada CSV; distribusi supertype dilewati.")

release_dates = pd.to_datetime(df['set.release_date'], format='%Y/%m/%d', errors='coerce')
print('Rentang rilis:', release_dates.min().date(), 'sampai', release_dates.max().date())
price_rows = []
for column in ['prices.cardmarket_trend', 'prices.tcgplayer_variants.holofoil.market']:
    values = pd.to_numeric(df[column], errors='coerce')
    price_rows.append({
        'field': column,
        'available': int(values.notna().sum()),
        'missing': int(values.isna().sum()),
        'coverage_percent': round(values.notna().mean() * 100, 2),
        'min': values.min(),
        'median': values.median(),
        'mean': values.mean(),
        'max': values.max(),
    })
display(pd.DataFrame(price_rows))

In [ ]:
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype=str).fillna('')
    manifest['width'] = pd.to_numeric(manifest['width'], errors='coerce')
    manifest['height'] = pd.to_numeric(manifest['height'], errors='coerce')
    manifest['file_size'] = pd.to_numeric(manifest['file_size'], errors='coerce')
    manifest['local_exists'] = manifest['local_path'].map(lambda value: Path(value).exists())
    manifest['image_valid'] = (
        manifest['status'].eq('downloaded')
        & manifest['local_exists']
        & manifest['width'].ge(100)
        & manifest['height'].ge(100)
        & manifest['file_size'].gt(0)
    )
    display(manifest['status'].value_counts().to_frame('count'))
    display(manifest[['width', 'height', 'file_size']].describe())
    print(f"Gambar valid lokal: {manifest['image_valid'].sum():,}/{len(manifest):,}")
else:
    manifest = pd.DataFrame(columns=['card_id', 'image_valid'])
    print('Manifest belum ada. Jalankan downloader terlebih dahulu.')

In [ ]:
# Missing harga atau metadata tidak otomatis menghapus kartu.
valid_ids = set(manifest.loc[manifest['image_valid'], 'card_id'])
clean_df = df[df['card_id'].astype(str).isin(valid_ids)].copy()
clean_df.to_csv(clean_csv_path, index=False)

if json_path.exists():
    with json_path.open(encoding='utf-8') as handle:
        json_records = json.load(handle)
    clean_records = [
        record for record in json_records
        if str(record.get('card_id')) in valid_ids
    ]
    with clean_json_path.open('w', encoding='utf-8') as handle:
        json.dump(clean_records, handle, ensure_ascii=False, indent=2)
else:
    clean_records = []

print(f'Baris metadata awal: {len(df):,}')
print(f'Baris clean berbasis gambar lokal: {len(clean_df):,}')
print(f'Drop: {len(df) - len(clean_df):,}')
print(f'CSV clean: {clean_csv_path}')
print(f'JSON clean: {clean_json_path} ({len(clean_records):,} record)')